# 00 · Synthetic input data (`data/dummy_data*.csv`)

The Fitbit cohort cannot be redistributed. This notebook writes two small **synthetic**
user-month tables in the public input schema so that `01_cohort_stats.ipynb`,
`02_modelling.ipynb` and `03_model_results.ipynb` can be run end to end:

| File | Metric | Columns |
| :--- | :--- | :--- |
| `dummy_data.csv` | resting heart rate | `id, month, age, sex, country, latitude, longitude, temperature, temperature min, temperature max, day light, resting heart rate, region` |
| `dummy_data_hrv.csv` | heart rate variability | same, with `heart rate variability` as target |

Run all cells once. Files are written to the data root (`./data` next to this repository,
or `$WPM_DATA_DIR`); existing files are left untouched unless `OVERWRITE = True`.
**Nothing estimated from these files is meaningful** — they only exercise the code paths.
To analyse your own data, write it in the same schema (see `data/README.md`) and skip
this notebook.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import sys

import pandas as pd

sys.path.insert(
    0, os.path.abspath('../src')
)  # library modules (public release)
import cohort_io as cio
import file_io as fio
import generate_dummy_data as gdd

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
N_USERS = 300  # RHR users; the HRV cohort is a ~2/3 subset
SEED = 0  # fixed seed -> identical files on every run
OVERWRITE = False  # True regenerates existing files

data_dir = fio.get_data_dir()
print('data root:', data_dir)

In [ ]:
targets = {
    m: os.path.join(data_dir, cio.PUBLIC_CSV_NAMES[m]) for m in ('rhr', 'hrv')
}
existing = [p for p in targets.values() if fio.file_exists(p)]
if existing and not OVERWRITE:
  print('already present (set OVERWRITE = True to regenerate):')
  for p in existing:
    print('  ', p)
else:
  written = gdd.write_dummy_data(data_dir, n_users=N_USERS, seed=SEED)

In [ ]:
# Quick look: schema check through the same loader the analysis notebooks use.
for metric in ('rhr', 'hrv'):
  df = cio.load_cohort(data_dir, metric)
  print(
      f'{metric}: {len(df):,} rows, {df["user_id"].nunique():,} users, '
      f'{df["activity_period"].min()} .. {df["activity_period"].max()}'
  )
df.head()

**Next:** [`01_cohort_stats.ipynb`](01_cohort_stats.ipynb) (model-free tables and figures),
then [`02_modelling.ipynb`](02_modelling.ipynb) once per metric and
[`03_model_results.ipynb`](03_model_results.ipynb).